### Task I: Quantum Computing Part
1) implement a simple quantum operation with Cirq or Pennylane
- With 5 qubits
- Apply Hadamard operation on every qubit
- Apply CNOT operation on (0, 1), (1,2), (2,3), (3,4)
- SWAP (0, 4)
- Rotate X with pi/2 on any qubit
- Plot the circuit
2) Implement a second circuit with a framework of your choice:
- Apply a Hadmard gate to the first qubit
- rotate the second qubit by pi/3 around X
- Apply Hadamard gate to the third and fourth qubit
- Perform a swap test between the states of the first and second qubit |q1 q2> and the third and fourth qubit |q3 q4>


In [3]:
import pennylane as qml
from pennylane import numpy as np
import cirq

### Circuit 1: Pennylane

In [4]:
# Define a 5-qubit device
dev = qml.device("default.qubit", wires=5)

@qml.qnode(dev)
def circuit1():
    # a) Hadamard on every qubit
    for i in range(5):
        qml.Hadamard(wires=i)

    # b) CNOT chain
    qml.CNOT(wires=[0,1])
    qml.CNOT(wires=[1,2])
    qml.CNOT(wires=[2,3])
    qml.CNOT(wires=[3,4])

    # c) SWAP (0,4)
    qml.SWAP(wires=[0,4])

    # d) RX(pi/2) on qubit 2 / wires = 1 since indexing starts from 0
    qml.RX(np.pi/2, wires=1)

    return qml.state()

# Draw the circuit
print(qml.draw(circuit1)())

0: ──H─╭●──────────╭SWAP───────────┤  State
1: ──H─╰X─╭●───────│──────RX(1.57)─┤  State
2: ──H────╰X─╭●────│───────────────┤  State
3: ──H───────╰X─╭●─│───────────────┤  State
4: ──H──────────╰X─╰SWAP───────────┤  State


### Circuit 2: Cirq

In [5]:
import numpy as np

# Define 6 qubits (4 for the test + 2 ancilla)
q1, q2, q3, q4, anc1, anc2 = cirq.LineQubit.range(6)

# Build circuit
circuit2 = cirq.Circuit()

# a) Hadamard on q1
circuit2.append(cirq.H(q1))

# b) RX(pi/3) on q2
circuit2.append(cirq.rx(np.pi/3)(q2))

# c) Hadamard on q3 and q4
circuit2.append([cirq.H(q3), cirq.H(q4)])

# d) Swap test between |q1 q2> and |q3 q4>
# Step 1: Hadamard on ancilla
circuit2.append(cirq.H(anc1))
circuit2.append(cirq.H(anc2))

# Step 2: Controlled-SWAP (Fredkin) between pairs
circuit2.append(cirq.CSWAP(anc1, q1, q2))
circuit2.append(cirq.CSWAP(anc2, q3, q4))

# Step 3: Hadamard on ancilla again
circuit2.append(cirq.H(anc1))
circuit2.append(cirq.H(anc2))

# Measure ancilla (result of swap test)
circuit2.append(cirq.measure(anc1, key="swap_test1"))
circuit2.append(cirq.measure(anc2, key="swap_test2"))

print(circuit2)

                   ┌──┐
0: ───H─────────────×───────────────────────────
                    │
1: ───Rx(0.333π)────×───────────────────────────
                    │
2: ───H─────────────┼×──────────────────────────
                    ││
3: ───H─────────────┼×──────────────────────────
                    ││
4: ───H─────────────@┼────H───M('swap_test1')───
                     │
5: ───H──────────────@────H───M('swap_test2')───
                   └──┘
